# Notebook 1 – Data Preprocessing Fundamentals

This notebook introduces the core concepts of data preprocessing — the foundation for every step that comes later in a machine learning project. Each concept is explained and then illustrated with real examples pulled from `customer_transactions_raw.csv`, a deliberately messy customer-transactions dataset.


In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
df = pd.read_csv('customer_transactions_raw.csv')
print("Shape:", df.shape)
df.head()

Shape: (1000, 12)


,customer_id,age,gender,annual_income,city,membership_type,purchase_amount,quantity,signup_date,payment_method,rating,notes
0,100508,43.0,female,80242.98,Bengaluru,Gold,99.13,1.0,08-22-2021,Debit Card,1,NaN
1,100819,41.0,Female,NaN,Delhi,Silver,113.23,1.0,16 Jun 2020,Net Banking,4,NaN
2,100453,61.0,Male,56285.40,Bengaluru,Gold,248.19,2.0,28 Sep 2020,Debit Card,4,NaN
3,100369,NaN,Male,81878.74,NaN,Silver,51.27,9.0,10 Sep 2023,Credit Card,4,NaN
4,100243,24.0,Male,100566.42,Delhi,Silver,97.97,1.0,07/01/2021,Credit Card,4,NaN


## 1. What is Data Preprocessing?

**Data preprocessing** is the set of steps taken to transform raw, messy, real-world data into a clean, structured, and consistent form that a machine learning algorithm can actually learn from. It sits between data collection and model building, and typically includes: handling missing values, removing duplicates, fixing invalid or inconsistent entries, correcting data types, encoding categories, scaling numbers, and engineering useful features.

Machine learning models don't understand raw text, inconsistent labels, or missing cells — preprocessing is the translation layer between "data as it exists in the world" and "data a model can consume."


In [2]:
df.dtypes

customer_id          int64
age                    str
gender                 str
annual_income      float64
city                   str
membership_type        str
purchase_amount        str
quantity           float64
signup_date            str
payment_method         str
rating               int64
notes              float64
dtype: object

Even a quick look at the dtypes above hints at preprocessing needs: `age` and `purchase_amount` are read as text (`object`/string) instead of numbers — a strong sign the raw values contain non-numeric characters that need to be cleaned before they can be used numerically.


## 2. Why is Data Preprocessing Required?

Real-world data is almost never analysis-ready. It is collected from multiple sources (forms, sensors, databases, manual entry), each with its own conventions, errors, and gaps. Without preprocessing:

- Models can crash on missing values or non-numeric text.
- Inconsistent labels (`Male` vs `male` vs `M`) get treated as different categories, diluting patterns.
- Invalid values (negative ages, impossible ratings) mislead statistics and models.
- Duplicate records bias the model toward over-represented customers.
- Scale differences between features (e.g. `annual_income` in the tens of thousands vs `rating` from 1–5) can make some algorithms overweight large-magnitude features.

Preprocessing exists to remove these obstacles **before** they distort analysis or model training.


In [3]:
print("Data types read as text that should be numeric:")
print(df[['age', 'purchase_amount']].dtypes)
print()
print("Sample raw values:")
print(df['age'].dropna().unique()[:5])
print(df['purchase_amount'].unique()[:5])


Data types read as text that should be numeric:
age                str
purchase_amount    str
dtype: object

Sample raw values:
<StringArray>
['43.0', '41.0', '61.0', '24.0', '39.0']
Length: 5, dtype: str
<StringArray>
['99.13', '113.23', '248.19', '51.27', '97.97']
Length: 5, dtype: str


Notice `purchase_amount` contains values like `$41.88` — the dollar sign forces the whole column to be read as text. `age` similarly hides a data problem we'll see shortly. Both are exactly the kind of issue preprocessing exists to fix.


## 3. Raw Data vs Clean Data

- **Raw data** is data exactly as collected — unfiltered, unvalidated, and often inconsistent. It may contain typos, mixed formats, missing entries, duplicates, and outright errors.
- **Clean data** has been checked, corrected, and standardized: consistent types, consistent category labels, validated ranges, no unresolved duplicates, and no unexplained missing values.

The dataset loaded above is raw. Let's look at a few rows that illustrate exactly what "raw" looks like in practice.


In [3]:
df.iloc[[0, 1, 3]]

,customer_id,age,gender,annual_income,city,membership_type,purchase_amount,quantity,signup_date,payment_method,rating,notes
0,100508,43.0,female,80242.98,Bengaluru,Gold,99.13,1.0,08-22-2021,Debit Card,1,NaN
1,100819,41.0,Female,NaN,Delhi,Silver,113.23,1.0,16 Jun 2020,Net Banking,4,NaN
3,100369,NaN,Male,81878.74,NaN,Silver,51.27,9.0,10 Sep 2023,Credit Card,4,NaN


Just in these three rows:
- Row 0: `city` has trailing whitespace (`"Bengaluru "`).
- Row 1: `annual_income` is missing.
- Row 3: `age` and `city` are both missing.

A **clean** version of this dataset would have trimmed whitespace, standardized categories, numeric types where appropriate, resolved or flagged missing values, and validated ranges — none of which is true yet.


## 4. Data Quality

**Data quality** is an umbrella term covering how fit-for-purpose a dataset is. It is usually broken down into measurable dimensions — completeness, consistency, accuracy, validity, and integrity (each covered individually below). A dataset is considered "high quality" when it scores well across all of these dimensions, not just one.

A quick multi-angle quality check:


In [4]:
quality_report = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().mean() * 100).round(2),
    'dtype': df.dtypes.astype(str)
})
quality_report

,missing_count,missing_pct,dtype
customer_id,0,0.0,int64
age,44,4.4,str
gender,90,9.0,str
annual_income,57,5.7,float64
city,153,15.3,str
membership_type,63,6.3,str
purchase_amount,0,0.0,str
quantity,20,2.0,float64
signup_date,16,1.6,str
payment_method,34,3.4,str


This single table already touches multiple quality dimensions at once — dtype problems (`age`, `purchase_amount` as text), and completeness problems (missing counts per column). The sections below unpack each dimension individually.


## 5. Data Consistency

**Data consistency** means the same real-world value is always represented the same way throughout the dataset. Inconsistent data doesn't necessarily contain *wrong* information — it contains the *same* information written in different, incompatible forms, which fragments what should be one category into several.


In [5]:
print("gender values:", sorted(df['gender'].dropna().unique().tolist()))
print()
print("city values:", sorted(df['city'].dropna().unique().tolist()))

gender values: ['F', 'Female', 'M', 'MALE', 'Male', 'female', 'male']

city values: [' Bengaluru', ' Chennai', ' Delhi', ' Hyderabad', ' Mumbai', 'BENGALURU', 'Bengaluru', 'Bengaluru ', 'CHENNAI', 'Chennai', 'Chennai ', 'DELHI', 'Delhi', 'Delhi ', 'HYDERABAD', 'Hyderabad', 'Hyderabad ', 'MUMBAI', 'Mumbai', 'Mumbai ', 'bengaluru', 'chennai', 'delhi', 'hyderabad', 'mumbai']


`gender` has only two real categories (male/female) but appears as `{Female, female, F, Male, male, M, MALE}` — seven distinct strings for two concepts. `city` has similar problems: `"Bengaluru"`, `"Bengaluru "` (trailing space), `"chennai"`, `"Chennai"`, `"delhi"`, `" Delhi"` (leading space) — casing and whitespace are silently creating duplicate categories. This is a textbook consistency problem.


## 6. Data Completeness

**Data completeness** measures how much of the expected data is actually present — i.e., how many values are missing. Completeness problems range from a few scattered missing cells to entire columns that are unusable.


In [6]:
completeness = (1 - df.isnull().mean()) * 100
completeness.sort_values().to_frame(name='completeness_pct')

,completeness_pct
notes,0.0
city,84.7
gender,91.0
membership_type,93.7
annual_income,94.3
age,95.6
payment_method,96.6
quantity,98.0
signup_date,98.4
customer_id,100.0


`notes` is 0% complete — every single value is missing, making the column entirely uninformative in its current form (a candidate for dropping). Other columns like `city`, `gender`, and `age` have partial gaps (roughly 4–15% missing) that are more typical and usually addressed through imputation rather than removal.


## 7. Data Accuracy

**Data accuracy** asks whether the recorded value correctly reflects the real-world value it's supposed to represent. A value can be present, of the right type, and even within a plausible range, and still be *inaccurate* — for example, a rating of `10` on what looks like a 1–5 scale, or an age that no living customer could actually have.


In [7]:
print("Rating value counts:")
print(df['rating'].value_counts().sort_index())

Rating value counts:
rating
-1       2
 0       1
 1     210
 2     187
 3     207
 4     198
 5     190
 7       3
 10      2
Name: count, dtype: int64


In [8]:
print("Ages above 100:")
print(df[df['age'].astype(str) == '150.0'][['customer_id', 'age']])

Ages above 100:
     customer_id    age
40        100873  150.0
889       100303  150.0


The `rating` column contains `-1` and `10`, both implausible if the intended scale is 1–5 (or even 1–10, a `-1` is still invalid). Similarly, an `age` of `150` is not a realistic human age. These are accuracy problems — the data was recorded, but the recorded value is very unlikely to be true, and needs verification or correction rather than a simple type fix.


## 8. Data Validity

**Data validity** checks whether values conform to the expected format, type, or business rule for that field — regardless of whether the value is "true." A date stored in five different formats is a validity problem even if every individual date is correct; a `payment_method` of `crypto` may be invalid simply because it's not a payment method the business actually supports.


In [9]:
print("Distinct signup_date formats (sample):")
print(df['signup_date'].dropna().unique()[:10])

Distinct signup_date formats (sample):
<ArrowStringArray>
[ '08-22-2021', '16 Jun 2020', '28 Sep 2020', '10 Sep 2023',  '07/01/2021',
  '01-19-2021', '30 Jan 2023',  '07-21-2023',  '13/09/2019',  '07/10/2020']
Length: 10, dtype: str


In [10]:
print("payment_method values:", df['payment_method'].dropna().unique())

payment_method values: <ArrowStringArray>
['Debit Card', 'Net Banking', 'Credit Card', 'UPI', 'COD', 'crypto']
Length: 6, dtype: str


`signup_date` mixes at least three different date formats (`MM-DD-YYYY`, `DD Mon YYYY`, `DD/MM/YYYY`) — a parser expecting one format will silently mis-read or fail on the others. `payment_method` includes `crypto`, which doesn't match the casing/style convention of the other methods (`Debit Card`, `Net Banking`, etc.) and may fall outside the set of officially supported payment options — worth confirming against business rules.


## 9. Data Integrity

**Data integrity** refers to the overall reliability and structural soundness of the data across the dataset — no unintended duplicate records, consistent relationships between fields, and no corruption introduced during storage or transfer. Integrity is broader than any single column; it's about whether the dataset *as a whole* can be trusted.


In [11]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated customer_id values:", df['customer_id'].duplicated().sum())

Fully duplicated rows: 50
Duplicated customer_id values: 50


50 duplicate `customer_id` values (matching the 50 fully duplicated rows) mean the same customer transaction appears to be recorded twice. If `customer_id` is meant to be unique per row, this is an integrity violation — left unresolved, these customers would be double-counted in any aggregate statistic or double-weighted during model training.


## 10. Data Leakage

**Data leakage** happens when information that would not be available at real prediction time is allowed to influence model training — causing the model to look artificially accurate during development but fail in production. Common sources include: including the target variable (or a proxy for it) as a feature, using future information to predict the past, or letting information from the test set leak into training (e.g. scaling using statistics computed on the full dataset before splitting).

**Example in this dataset's context:** if we were building a model to predict `membership_type` (e.g. who will upgrade to Gold), and we included `purchase_amount` computed *after* the upgrade happened, that would leak future information into a feature meant to predict the past. Similarly, if we compute the mean of `annual_income` across the *entire* dataset to fill missing values *before* splitting into train/test sets, statistics from the test set leak into training — a subtle but common form of leakage.

The key discipline against leakage: always split data into train/validation/test **first**, then fit any preprocessing step (imputation values, scalers, encoders) only on the training set, and apply those same fitted transformations to validation/test data.


## 11. Training Data

**Training data** is the portion of the dataset used to actually fit the model — the algorithm learns patterns, relationships, and parameters from this subset. It is typically the largest split (commonly 60–80% of the data).

## 12. Validation Data

**Validation data** is a held-out portion used *during development* to tune hyperparameters and compare candidate models, without touching the test set. It acts as a proxy for "unseen data" while the model is still being iterated on — if you keep adjusting the model based on validation performance, the model can start to indirectly "see" the validation set, so it should not be used for the final performance claim.

## 13. Test Data

**Test data** is a held-out portion used only once, at the very end, to report the model's final, unbiased performance. It should never influence any decision made during model development — not feature selection, not hyperparameter tuning, not preprocessing choices.


In [12]:
from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
print(f"Training set:   {len(train_df)} rows ({len(train_df)/len(df):.0%})")
print(f"Validation set: {len(val_df)} rows ({len(val_df)/len(df):.0%})")
print(f"Test set:       {len(test_df)} rows ({len(test_df)/len(df):.0%})")

Training set:   700 rows (70%)
Validation set: 150 rows (15%)
Test set:       150 rows (15%)


**Important:** any preprocessing step that "learns" something from the data (e.g. the mean used for imputation, a scaler's min/max, an encoder's category list) must be fit on `train_df` only, then applied unchanged to `val_df` and `test_df`. Fitting on the full dataset before splitting is a common cause of data leakage.


## 14. Preprocessing Pipeline

A **preprocessing pipeline** is a defined, repeatable sequence of transformation steps applied to data — so that the exact same operations, fitted the same way, can be applied to new data (validation, test, or future production data) without manually repeating each step by hand. Pipelines make preprocessing:

- **Reproducible** — the same steps run the same way every time.
- **Leak-safe** — steps are fit on training data and applied consistently elsewhere.
- **Maintainable** — the whole workflow lives in one place instead of scattered notebook cells.

In scikit-learn, this is often implemented with `Pipeline` and `ColumnTransformer` objects that chain imputers, encoders, and scalers together into a single object that can be `.fit()` on training data and `.transform()` on anything else.


## 15. The Complete Preprocessing Workflow

Putting all of the above together, a typical end-to-end preprocessing workflow looks like this:

1. **Data collection** — gather raw data from its source(s) (as already done for `customer_transactions_raw.csv`).
2. **Initial inspection** — check shape, dtypes, and a sample of rows to understand what you're working with.
3. **Data quality assessment** — systematically check completeness (missing values), consistency (label variants), accuracy (implausible values), validity (format/business-rule conformance), and integrity (duplicates, relationships).
4. **Train/validation/test split** — split the raw (or minimally cleaned) data *before* fitting any preprocessing step that learns from the data, to prevent leakage.
5. **Missing value treatment** — impute or drop, using strategies decided from training data only.
6. **Duplicate removal** — drop exact or logical duplicates (e.g. duplicate `customer_id`s).
7. **Fixing invalid/inaccurate values** — correct or remove impossible values (e.g. `rating = -1`, `age = 150`), and standardize formats (e.g. unify `signup_date` formats, strip `$` from `purchase_amount`).
8. **Consistency normalization** — standardize categorical labels (casing, whitespace, abbreviations) so each real-world category maps to exactly one value.
9. **Data type correction** — cast columns to their correct types (numeric columns as numbers, categories as categorical, dates as datetime).
10. **Feature encoding** — convert categorical variables into a numeric form models can use (one-hot, label, or target encoding).
11. **Feature scaling** — standardize or normalize numeric features so no single feature dominates due to scale alone.
12. **Feature engineering** — create new, more informative features from existing ones (e.g. tenure buckets, ratios, date-derived features).
13. **Final validation** — re-check the cleaned dataset's quality dimensions to confirm the issues found in step 3 have actually been resolved.
14. **Model-ready dataset** — hand off `train_df` / `val_df` / `test_df`, consistently preprocessed via a fitted pipeline, to the modeling stage.

This exact workflow (steps 5–12 in particular) is what the upcoming preprocessing notebooks will apply, in order, to this dataset.
